# Error analysis

Where does the model lose BERTScore F1 on dev, and why? Needs the predictions file and its `.scores.jsonl` (written by `evaluate()`), plus the dev gold answers.

The released data has **no `country` / `category` fields**, so the splits below use a question-type rule and a place-name proxy, both copied from `shared/notebooks/00_EDA.ipynb`.

In [ ]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "shared"))
from data import RESULTS_DIR, load_jsonl, load_split, preds_path

pd.set_option("display.max_colwidth", 200)

MODEL = "Qwen3-VL-4B-Instruct"
PROMPT_ID = "gold_style_v1"
PREDS = preds_path("t2_text", "dev", MODEL, PROMPT_ID)
SCORES = PREDS.with_suffix(".scores.jsonl")
print(PREDS)

In [ ]:
if not PREDS.exists():
    raise FileNotFoundError(f"{PREDS} not found: download the predictions first")
if not SCORES.exists():
    raise FileNotFoundError(
        f"{SCORES} not found. Score the predictions once (run from shared/):\n"
        f"  from metrics import evaluate; evaluate({str(PREDS)!r}, 'dev')")

gold = pd.DataFrame(load_split("qa", "dev")).drop_duplicates("id", keep="last")  # same as evaluate(): one row per id
gold = gold[["id", "question", "answer"]].rename(columns={"answer": "gold"})
preds = pd.DataFrame(load_jsonl(PREDS)).drop_duplicates("id", keep="last")[["id", "answer"]].rename(columns={"answer": "pred"})
scores = pd.DataFrame(load_jsonl(SCORES))[["id", "p", "r", "f1"]]

df = gold.merge(preds, on="id").merge(scores, on="id")
df["pred"] = df["pred"].fillna("")
print(f"{len(df)} scored examples; gold ids without a prediction: {len(gold) - len(df)}")

## 1. Overview

In [ ]:
print(f"{MODEL} / {PROMPT_ID}: n={len(df)}  F1={df.f1.mean():.4f}  P={df.p.mean():.4f}  R={df.r.mean():.4f}")
print("F1 quantiles:"); display(df["f1"].describe(percentiles=[.05, .25, .5, .75, .95]).round(4).to_frame().T)

In [ ]:
rows = [json.load(open(p)) for p in sorted(RESULTS_DIR.glob("t2_text_dev_*.json"))]
runs = pd.DataFrame(rows)
if len(runs) > 1:
    display(runs[["preds", "n", "bertscore_f1", "bertscore_p", "bertscore_r"]].round(4))
else:
    print("Only one run in results/, nothing to compare.")

## 2. Answer length

In [ ]:
df["g_words"] = df["gold"].str.split().str.len()
df["p_words"] = df["pred"].str.split().str.len()
df["ratio"] = df["p_words"] / df["g_words"].clip(lower=1)
display(df[["g_words", "p_words"]].agg(["mean", "median"]).round(1))

fig, ax = plt.subplots(figsize=(6, 3.5))
bins = range(0, int(max(df.g_words.max(), df.p_words.max())) + 3, 2)
ax.hist(df["g_words"], bins=bins, alpha=.5, label="gold")
ax.hist(df["p_words"], bins=bins, alpha=.5, label="pred")
ax.set(xlabel="words", ylabel="examples"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(df["ratio"], df["f1"], s=6, alpha=.4)
ax.set(xlabel="pred words / gold words", ylabel="F1", xscale="log")
plt.tight_layout(); plt.show()

df["ratio_bin"] = pd.cut(df["ratio"], [0, .5, .8, 1.25, 2, np.inf], labels=["<0.5", "0.5-0.8", "0.8-1.25", "1.25-2", ">2"])
display(df.groupby("ratio_bin", observed=True)["f1"].agg(n="size", mean_f1="mean").round(4))

## 3. Question type

Same rule as the EDA notebook, plus the first two words of the question.

In [ ]:
def qtype(q):
    q = q.lower()
    for k, pat in [("significance/meaning", r"significan|symboli|meaning|represent"), ("cultural/traditional", r"cultur|tradition|heritage"),
                   ("identify (what is/which/who)", r"^(what|which|who)\b.*\b(is|are|name|type|kind)\b")]:
        if re.search(pat, q): return k
    return "other"

df["qtype"] = df["question"].map(qtype)
display(df.groupby("qtype")["f1"].agg(n="size", mean_f1="mean").round(4))

In [ ]:
df["opening"] = df["question"].str.split().str[:2].str.join(" ")
top = df.groupby("opening")["f1"].agg(n="size", mean_f1="mean").round(4)
display(top[top["n"] >= 10].sort_values("n", ascending=False).head(10))

## 4. Place-name proxy for country

There is no country field. The proxy is the list of country and city names from the EDA notebook, matched in the **gold** answer (the question alone rarely names the place). It is a rough guess: unmatched examples may still be about some country, and one example can name several.

In [ ]:
PLACES = {
    "Egypt": ["egypt", "cairo", "giza", "luxor", "alexandria", "aswan"],
    "UAE": ["uae", "emirat", "dubai", "abu dhabi", "sharjah"],
    "Saudi Arabia": ["saudi", "riyadh", "mecca", "makkah", "medina", "jeddah"],
    "Qatar": ["qatar", "doha"], "Bahrain": ["bahrain", "manama"], "Kuwait": ["kuwait"],
    "Oman": ["oman", "muscat"], "Yemen": ["yemen", "sanaa", "aden"],
    "Iraq": ["iraq", "baghdad", "mosul", "najaf", "karbala", "basra", "erbil"],
    "Syria": ["syria", "damascus", "aleppo", "palmyra"],
    "Lebanon": ["leban", "beirut", "byblos", "baalbek"], "Jordan": ["jordan", "amman", "petra"],
    "Palestine": ["palestin", "gaza", "jerusalem", "ramallah", "bethlehem"],
    "Morocco": ["morocc", "marrakech", "fez", "casablanca", "rabat"],
    "Algeria": ["alger", "algiers"], "Tunisia": ["tunis"], "Libya": ["libya", "tripoli", "benghazi"],
    "Sudan": ["sudan", "khartoum"],
}
def guess_countries(text):
    t = text.lower()
    return [c for c, keys in PLACES.items() if any(re.search(r"\b" + re.escape(k), t) for k in keys)]

df["countries"] = df["gold"].map(guess_countries)
df["names_place"] = df["countries"].str.len() > 0
display(df.groupby("names_place")["f1"].agg(n="size", mean_f1="mean").round(4))

In [ ]:
by_country = df.explode("countries").dropna(subset=["countries"]).groupby("countries")["f1"].agg(n="size", mean_f1="mean").round(4)
display(by_country[by_country["n"] >= 5].sort_values("mean_f1"))

## 5. Worst 20 answers

Tag each with one label:

- `wrong entity`: names a different landmark, dish, person or country than the gold.
- `right entity but generic`: the entity is right, but the answer is vague or lacks the cultural detail.
- `too long`: far longer than gold (extra sentences, explanations).
- `refusal`: says it cannot tell, or declines to answer.
- `other`: anything else (note why in a comment).

In [ ]:
worst = df.nsmallest(20, "f1")[["id", "question", "gold", "pred", "f1"]].reset_index(drop=True)
worst["tag"] = ""
display(worst.drop(columns="id").round(3))

In [ ]:
# copy this printout into the next cell and fill in a tag per id
print("TAGS = {")
for i, r in worst.iterrows():
    print(f'    "{r.id}": "",  # {i}')
print("}")

In [ ]:
VOCAB = ["wrong entity", "right entity but generic", "too long", "refusal", "other"]
TAGS = {}  # {id: tag}, paste the template from the cell above

bad = {i: t for i, t in TAGS.items() if t not in VOCAB}
assert not bad, f"unknown tags (use one of {VOCAB}): {bad}"
worst["tag"] = worst["id"].map(TAGS).fillna("untagged")
display(worst["tag"].value_counts().to_frame("count"))

## 6. Best 5 answers (midterm examples)

In [ ]:
for _, r in df.nlargest(5, "f1").iterrows():
    print(f"F1 {r.f1:.3f}\nQ:    {r.question}\nGold: {r.gold}\nPred: {r.pred}\n")